In [1]:
# Parameters
runfold = "/nfs/scistore26/saricgrp/fhorvath/treadmilling-functions/test_simulations/data/run_kswitch0.2_rdis1/runfiles"
rundir = "/nfs/scistore26/saricgrp/fhorvath/treadmilling-functions/test_simulations/data/run_kswitch0.2_rdis1/runfiles"
num_cores = 10
delete = 0


In [2]:
import os
import pandas as pd
import json
from tqdm.notebook import tqdm
import functions as fct
import pickle

Septal bins set: 19 y-bins, -44.5 to 41.0 nm in steps of 4.5 nm (0.9 sim units)


In [3]:
try: 
    delete
except NameError:
    print("not passed via papermill, assigning default value")
    delete = True
delete


0

In [4]:
try: 
    inverty
except NameError:
    print("not passed via papermill, assigning default value")
    inverty = False
inverty


not passed via papermill, assigning default value


False

In [5]:
try: 
    downsample
except NameError:
    print("not passed via papermill, assigning default value")
    downsample = None
downsample


not passed via papermill, assigning default value


In [6]:
try: 
    columns
except NameError:
    print("not passed via papermill, assigning default value")
    columns = None
columns


not passed via papermill, assigning default value


In [7]:
if not os.path.exists(runfold + "/df.pkl.gz"):

    df = fct.xyz_reader.read_xyz(runfold+"/")
    if columns:
        fct.utils.compress_pickle(df[columns], runfold + "/df.pkl.gz")
    else:
        fct.utils.compress_pickle(df, runfold + "/df.pkl.gz")

Streaming XYZ:   0%|          | 0.00/370k [00:00<?, ?B/s]

In [8]:
if inverty:
    df.loc[:, "y"] = -df.loc[:, "y"]

In [9]:
%cd {runfold}

/nfs/scistore26/saricgrp/fhorvath/treadmilling-functions/test_simulations/data/run_kswitch0.2_rdis1/runfiles


In [10]:
if downsample:
    if downsample==10:
        ! /nfs/scistore26/saricgrp/fhorvath/0__treadmilling/bashlib/bin/xyz_every1000th.sh
    elif downsample==100:
        ! /nfs/scistore26/saricgrp/fhorvath/0__treadmilling/bashlib/bin/xyz_every10000th.sh
    else:
        ! /nfs/scistore26/saricgrp/fhorvath/0__treadmilling/bashlib/bin/xyz_every10000th.sh

In [11]:
if os.path.exists("bonds.dump"):
    bonds = fct.xyz_reader.read_xyz(filename="bonds.dump")
    ## discard type-1 bonds
    bonds = bonds[bonds["c_cBonds[3]"]!=1]
    fct.utils.compress_pickle(bonds, runfold + "/bonds.pkl.gz")
    os.remove("bonds.dump")

In [12]:
import gzip
import pickle
import os
if delete:
    pkl_file = "df.pkl.gz"
    file_to_remove = "output.xyz"  # change to whichever file you want deleted

    def is_valid_pkl_gz(filename):
        try:
            with gzip.open(filename, "rb") as f:
                _ = pickle.load(f)
            return True
        except Exception as e:
            print(f"File {filename} is not valid: {e}")
            return False

    if is_valid_pkl_gz(pkl_file):
        if os.path.exists(file_to_remove):
            os.remove(file_to_remove)
            print(f"Deleted: {file_to_remove}")
        else:
            print(f"File to delete does not exist: {file_to_remove}")
    else:
        print("Not deleting anything because file is not valid.")
